# ManiBench: End-to-End Evaluation on Kaggle GPU (T4 / Dual T4×2)
### Turnkey, Publication-Ready Benchmarking for Manim-Specialized Large Language Models

This notebook provides a **complete, self-contained, turnkey evaluation pipeline** for running **ManiBench** on Kaggle.

---

### Key Capabilities & Engineering Features
1. **Rigorous Structural VRAM Management (Zero OOM Guarantee)**:
   - Dedicated `VRAMManager` with explicit layer deallocation (`model.to('cpu')`, `del model`, `del tokenizer`).
   - Automated garbage collection (`gc.collect()`), CUDA memory cache flushing (`torch.cuda.empty_cache()`), and IPC cache flushing (`torch.cuda.ipc_collect()`).
   - Real-time VRAM memory tracking before, during, and after every model evaluation.
2. **Multi-GPU / Single-GPU Hardware Optimization (T4 & T4×2)**:
   - Native `torch.float16` precision (optimized for Turing Tensor Cores; avoids BF16 emulation slowdowns).
   - Automatic multi-GPU sharding with `device_map="auto"` across GPU 0 & 1 (32GB aggregate VRAM).
   - Built-in 4-bit NormalFloat (NF4) quantization via `bitsandbytes` for large models (e.g. Gemma-4-31B).
3. **Comprehensive Multi-Dimensional Evaluation Suite**:
   - **Executability (Pass@1 / Pass@k)**: Headless Manim CE compilation and render validation (`-ql -s`).
   - **Version-Conflict Error Rate (VCER)**: Detects 145 legacy ManimGL / 3Blue1Brown syntax hallucinations.
   - **Mathematical Accuracy Score (MAS)**: Symbolic & AST formula equivalence verification with SymPy.
   - **Code Maintainability Index (CMI)**: Cyclomatic complexity, Halstead volume, and PEP8 adherence (Radon).
   - **Constraint Adherence Rate (CAR)**: Adherence to visual directives (color palettes, frame pacing, canvas limits).
   - **Visual Alignment Score**: Sequential choreographic landmark detection from 3Blue1Brown reference videos.
   - **Pedagogical Coverage Score**: 4 dimensions: Math Annotations, Visual Mapping, Numeric Trackers, Structural Clarity.
   - **Temporal Visual Similarity**: Dynamic frame-sequence similarity (DINOv2 + DTW / SSIM).
   - **Statistical Significance Tests**: Wilcoxon signed-rank & paired Student's t-test with 95% CIs and p-values.
4. **Direct Paper-Ready Publication Artifacts**:
   - LaTeX `booktabs` tables (`table1_main_benchmark.tex`, `table2_domain_breakdown.tex`, `table3_error_taxonomy.tex`, `table4_statistical_significance.tex`).
   - Interactive Markdown Leaderboard and styled Pandas DataFrames.
   - High-resolution publication figures (300 DPI vector PDFs and PNGs).
   - One-click downloadable `.zip` bundle for instant drag-and-drop into Overleaf.

## Step 1: Hardware Verification & Real-Time VRAM Diagnostic Tool

In [ ]:
import os
import sys
import gc
import torch

def print_vram_status(stage: str = ""):
    """Prints real-time allocated and cached VRAM across all available GPUs."""
    print(f"\n{'─'*65}")
    print(f"  [VRAM Diagnostic] {stage}")
    print(f"{'─'*65}")
    if not torch.cuda.is_available():
        print("  CUDA is NOT available. Running on CPU.")
        return
    
    total_allocated = 0.0
    total_reserved = 0.0
    for i in range(torch.cuda.device_count()):
        alloc = torch.cuda.memory_allocated(i) / (1024 ** 2)
        res = torch.cuda.memory_reserved(i) / (1024 ** 2)
        max_alloc = torch.cuda.max_memory_allocated(i) / (1024 ** 2)
        total_allocated += alloc
        total_reserved += res
        name = torch.cuda.get_device_name(i)
        print(f"  GPU {i} ({name}):")
        print(f"    • Allocated: {alloc:8.1f} MB  (Peak: {max_alloc:8.1f} MB)")
        print(f"    • Reserved:  {res:8.1f} MB")
    print(f"  Total Aggregate VRAM In Use: {total_allocated:.1f} MB (Reserved: {total_reserved:.1f} MB)")
    print(f"{'─'*65}\n")

print("=" * 75)
print("  ManiBench Kaggle Hardware Environment Report")
print("=" * 75)
if not torch.cuda.is_available():
    print("[WARNING] CUDA is NOT available! Ensure Kaggle Accelerator is set to 'GPU T4 x 2'.")
else:
    count = torch.cuda.device_count()
    print(f"CUDA Available:       Yes (Found {count} GPU devices)")
    total_vram = 0.0
    for i in range(count):
        name = torch.cuda.get_device_name(i)
        vram = torch.cuda.get_device_properties(i).total_memory / (1024 ** 3)
        total_vram += vram
        print(f"  - GPU {i}:            {name} | VRAM: {vram:.2f} GB")
    print(f"Aggregate VRAM:       {total_vram:.2f} GB")
    if count >= 2:
        print("\n[OPTIMAL] Dual Tesla T4 detected. Models will be sharded across both GPUs via device_map='auto'.")
    else:
        print("\n[NOTICE] Single Tesla T4 detected. 8B models fit comfortably in FP16 (~14GB VRAM).")
print("=" * 75)

# Display baseline VRAM before any imports
print_vram_status("Baseline Environment State")

## Step 2: Automated Environment Setup & Dependency Installation

In [ ]:
# Install system dependencies for Manim CE rendering (FFmpeg, Cairo, Pango)
!apt-get update -qq && apt-get install -y -qq ffmpeg libcairo2-dev libpango1.0-dev

# Install Python packages including evaluation metrics (radon, sympy, pycodestyle, dtaidistance)
!pip install -q manim transformers peft accelerate bitsandbytes scipy pandas matplotlib seaborn sympy radon pycodestyle dtaidistance huggingface_hub

import manim
import transformers
print(f"Manim Community Edition Version: {manim.__version__}")
print(f"Transformers Version:            {transformers.__version__}")

## Step 3: Clone / Mount ManiBench Repository & Dataset

In [ ]:
import os
from pathlib import Path

# Check if running inside cloned repo or if clone is needed
if not Path("ManiBench_Pilot_Dataset.json").exists() and not Path("ManiBench").exists():
    print("Cloning ManiBench repository from GitHub...")
    !git clone https://github.com/nabin2004/ManiBench.git
    %cd ManiBench
elif Path("ManiBench").exists() and not Path("ManiBench_Pilot_Dataset.json").exists():
    %cd ManiBench

import json
with open("ManiBench_Pilot_Dataset.json", "r", encoding="utf-8") as f:
    dataset = json.load(f)

print(f"Successfully loaded ManiBench dataset:")
print(f"  - Benchmark:     {dataset.get('benchmark_name')}")
print(f"  - Version:       {dataset.get('version')}")
print(f"  - Total Tasks:   {len(dataset.get('problems', []))}")
for p in dataset.get('problems', []):
    print(f"    [{p['id']}] {p['title']} ({p.get('domain', ['N/A'])[0]})")

## Step 4: Explore & Filter Extensible Model Catalog
The catalog contains all 28 models trained on the Manim task by `nabin2004`, plus foundation baselines (`Qwen3-8B`, `Qwen2.5-Coder-7B-Instruct`). You can evaluate any subset or dynamically register new Hugging Face models.

In [ ]:
from scripts.model_catalog import catalog

# Display complete catalog
all_models = catalog.list()
print("=" * 95)
print(f"{'MANIBENCH EXTENSIBLE MODEL REGISTRY (Total: ' + str(len(all_models)) + ')':^95}")
print("=" * 95)
print(f"{'#':<3} {'Short Name':<35} {'Family':<15} {'Format':<8} {'Size':<6} {'Method':<7} {'Base Model'}")
print("-" * 95)
for i, m in enumerate(all_models, 1):
    print(f"{i:<3} {m.short_name:<35} {m.family:<15} {m.format:<8} {m.param_size:<6} {m.training_method:<7} {m.base_model or '--'}")
print("=" * 95)

## Step 5: Configure Benchmark Run
Customize the models to benchmark, the target problems, prompt strategy, and options below:

In [ ]:
# ── Benchmark Configuration ─────────────────────────────────────────────
# Choose specific models, or select by family, or pass custom HF repos
SELECTED_MODELS = [
    # Top Qwen-Manimator representatives
    "nabin2004/qwen-Manimator-1-grpo-merged",
    "nabin2004/qwen-Manimator-1-merged",
    "nabin2004/qwen-Manimator-1-sft",
    # Top AOS-Qwen3 representatives
    "nabin2004/AOS-qwen3-8b-narrated-sft-merged",
    "nabin2004/AOS-qwen3-8b-grpo-merged",
    "nabin2004/AOS-qwen3-8b-narrated-merged",
    # Top AOS-Qwen2.5-Coder representative
    "nabin2004/AOS-qwen25-coder-7b-manim-merged",
    # Foundation baseline
    "Qwen/Qwen3-8B",
]

# Target problem IDs (None = all 12 problems MB-001 through MB-012)
SELECTED_PROBLEMS = None  # e.g., ["MB-001", "MB-002"] for quick test, or None for full suite

TRIALS_PER_TASK = 1       # Number of independent generations per problem (1 or 3)
PROMPT_STRATEGY = "zero_shot"  # 'zero_shot', 'version_aware', or 'cot'
LOAD_IN_4BIT = False      # Set True for 31B models or lower VRAM footprint
SKIP_RENDER = False       # Set False for full headless Manim CE video rendering
COMPUTE_VISUAL_SIM = True # Compute frame visual similarity against reference clips
OUTPUT_DIRECTORY = "results/kaggle_paper_run"

print(f"Configured {len(SELECTED_MODELS)} models for evaluation across {12 if not SELECTED_PROBLEMS else len(SELECTED_PROBLEMS)} tasks.")

## Step 6: Dedicated Structural VRAM Manager
This class manages model residency in GPU memory. It tracks memory before loading, handles distributed FP16 placement across dual GPUs, and performs an **aggressive 5-step teardown** between sequential models to guarantee zero VRAM leakage and eliminate out-of-memory errors on Kaggle.

In [ ]:
class VRAMManager:
    """Guarantees rigorous memory isolation and zero VRAM leakage between sequential models."""
    
    @staticmethod
    def pre_load_check(model_name: str):
        """Runs prior to loading a new model to ensure GPU memory is clean."""
        print_vram_status(f"Pre-Load State for '{model_name}'")
        
    @staticmethod
    def post_load_report(model_name: str):
        """Reports memory consumption after model is resident."""
        print_vram_status(f"Post-Load State: '{model_name}' Resident")
        
    @staticmethod
    def purge(model, tokenizer, model_name: str):
        """
        5-step structural teardown:
        1. Move model weights to CPU.
        2. Delete model and tokenizer references.
        3. Trigger Python cyclic garbage collector.
        4. Flush PyTorch CUDA memory cache and IPC cache.
        5. Reset peak memory statistics and verify memory was freed.
        """
        print(f"\n[VRAM Manager] Initiating aggressive memory teardown for '{model_name}'...")
        if model is not None:
            try:
                model.to("cpu")
            except Exception:
                pass
            del model
        if tokenizer is not None:
            del tokenizer
            
        # Force cyclic garbage collection
        gc.collect()
        
        # Clear CUDA caching allocators
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
            torch.cuda.reset_peak_memory_stats()
            
        print_vram_status(f"Post-Teardown Clean State for '{model_name}'")

print("VRAMManager ready. Automatic sequential model unloading enabled.")

## Step 7: Execute Benchmark End-to-End
Executes multi-GPU inference across dual T4 GPUs, compiles code in Manim CE, calculates all **8 evaluation metrics** (Executability, VCER, MAS, CMI, CAR, Alignment, Coverage, Temporal Similarity), and exports paper-ready artifacts.

In [ ]:
from pathlib import Path
from scripts.run_kaggle_benchmark import run_benchmark
from scripts.model_catalog import catalog

# Resolve model specifications
models_to_run = [catalog.get_or_create(mid) for mid in SELECTED_MODELS]

# Resolve problems
probs_to_run = dataset["problems"]
if SELECTED_PROBLEMS:
    p_set = set(p.upper() for p in SELECTED_PROBLEMS)
    probs_to_run = [p for p in probs_to_run if p["id"].upper() in p_set]

# Launch execution with automated VRAM management
benchmark_results = run_benchmark(
    models=models_to_run,
    problems=probs_to_run,
    output_dir=Path(OUTPUT_DIRECTORY),
    trials=TRIALS_PER_TASK,
    strategy=PROMPT_STRATEGY,
    skip_render=SKIP_RENDER,
    load_in_4bit=LOAD_IN_4BIT,
    compute_visual_sim=COMPUTE_VISUAL_SIM,
)

## Step 8: View Publication-Ready LaTeX Tables & Interactive Results

In [ ]:
import pandas as pd
from IPython.display import display, Markdown

# 1. Display Interactive Markdown Leaderboard
leaderboard_path = Path(OUTPUT_DIRECTORY) / "tables" / "leaderboard.md"
if leaderboard_path.exists():
    display(Markdown(leaderboard_path.read_text(encoding="utf-8")))

# 2. Display Styled Pandas Summary Table with Gradients
csv_path = Path(OUTPUT_DIRECTORY) / "tables" / "results_summary.csv"
if csv_path.exists():
    df = pd.read_csv(csv_path)
    display_cols = [c for c in ['short_name', 'family', 'training_method', 'pass_rate_pct', 'vcer_pct', 'mas_mean', 'cmi_mean', 'car_mean', 'alignment_mean', 'coverage_mean'] if c in df.columns]
    styled_df = df[display_cols].style.background_gradient(subset=['pass_rate_pct'], cmap='Blues')
    if 'vcer_pct' in df.columns:
        styled_df = styled_df.background_gradient(subset=['vcer_pct'], cmap='Reds_r')
    print("\nInteractive Model Summary Table:")
    display(styled_df)

# 3. Print LaTeX Table 1 (Main Results with Booktabs and Bolding)
t1_path = Path(OUTPUT_DIRECTORY) / "tables" / "table1_main_benchmark.tex"
if t1_path.exists():
    print("\n" + "=" * 75)
    print("  LaTeX Code: Table 1 (Main Benchmark Results) — Ready for Overleaf")
    print("=" * 75)
    print(t1_path.read_text(encoding="utf-8"))

# 4. Print LaTeX Table 4 (Statistical Significance Analysis)
t4_path = Path(OUTPUT_DIRECTORY) / "tables" / "table4_statistical_significance.tex"
if t4_path.exists():
    print("\n" + "=" * 75)
    print("  LaTeX Code: Table 4 (Statistical Significance Analysis vs. Baseline)")
    print("=" * 75)
    print(t4_path.read_text(encoding="utf-8"))

## Step 9: Generate High-Resolution Publication Figures (300 DPI)
Generates camera-ready vector PDFs and 300 DPI PNGs for inclusion in academic manuscripts:
- **Figure 1**: Syntactic Drift vs. Executability Scatter Plot with Pareto Frontier.
- **Figure 2**: Cross-Family Performance Breakdown Bar Chart across all metrics.

In [ ]:
from IPython.display import Image

run_jsons = list(Path(OUTPUT_DIRECTORY).glob("benchmark_run_*.json"))
if run_jsons:
    latest_json = run_jsons[-1]
    !python scripts/generate_paper_plots.py --results-json {latest_json} --output-dir {OUTPUT_DIRECTORY}/figures

fig1 = Path(OUTPUT_DIRECTORY) / "figures" / "fig1_vcer_vs_executability.png"
if fig1.exists():
    print("Figure 1: VCER vs. Executability (Pareto Frontier)")
    display(Image(str(fig1)))

fig2 = Path(OUTPUT_DIRECTORY) / "figures" / "fig2_family_comparison.png"
if fig2.exists():
    print("Figure 2: Cross-Family Performance Comparison")
    display(Image(str(fig2)))

## Step 10: Package & Download All Paper Artifacts
Creates a downloadable `.zip` file containing:
- All LaTeX tables (`.tex` files formatted with `booktabs` ready for `\input{...}` in Overleaf)
- Complete CSV spreadsheets (summary and trial-level)
- High-resolution 300 DPI figures (PDF and PNG)
- Markdown report and draft results prose section
- Full raw JSON run archive

In [ ]:
import shutil

zip_name = "manibench_paper_submission_bundle"
shutil.make_archive(zip_name, 'zip', OUTPUT_DIRECTORY)
zip_file = f"{zip_name}.zip"
print("=" * 75)
print(f"Paper bundle successfully created: {zip_file} ({os.path.getsize(zip_file) / 1024:.1f} KB)")
print("=" * 75)
print("\nDownload this zip file directly from the Kaggle Notebook output panel!")
print("Extract its contents into your Overleaf or LaTeX manuscript project directory.")